# Deep Learning — MLP Sector Classification

**PFA 2025-2026 | ENSAM Rabat**  
**Architecture :** Multi-Layer Perceptron (MLP)  
**Input :** 20 structured features (financement, pays, âge…)  
**Cible :** `primary_category_grouped` — 20 secteurs

## 1. Imports

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(42)
np.random.seed(42)

print(f'TensorFlow : {tf.__version__}')

## 2. Chargement des données

In [ ]:
pre   = pd.read_csv('preprocessed_startups.csv')
train = pd.read_csv('train_set.csv')
val   = pd.read_csv('val_set.csv')
test  = pd.read_csv('test_set.csv')

# Mapper le secteur depuis le fichier prétraité
sector_map = pre.set_index('name')['primary_category_grouped'].to_dict()

for split in [train, val, test]:
    split['sector'] = split['name'].map(sector_map)
    split.drop(split[split['sector'].isna()].index, inplace=True)
    split.drop(split[split['sector'] == 'Unknown'].index, inplace=True)

print(f'Train : {len(train):,}  |  Val : {len(val):,}  |  Test : {len(test):,}')
print(f'Secteurs : {train["sector"].nunique()}')

## 3. Préparation des features

In [ ]:
STRUCTURED = [
    'funding_log_scaled', 'funding_rounds_scaled', 'funding_per_round_log_scaled',
    'company_age_years_scaled', 'time_to_first_funding_days_scaled',
    'has_website', 'is_us_startup', 'has_multiple_categories',
    'country_AUS', 'country_CAN', 'country_CHN', 'country_DEU', 'country_ESP',
    'country_FRA', 'country_GBR', 'country_IND', 'country_ISR',
    'country_Other', 'country_USA'
]

for split in [train, val, test]:
    split[STRUCTURED] = split[STRUCTURED].fillna(0)

# founded_year nécessite un StandardScaler (pas encore normalisé)
scaler = StandardScaler()
fy_train = scaler.fit_transform(train[['founded_year']]).ravel()
fy_val   = scaler.transform(val[['founded_year']]).ravel()
fy_test  = scaler.transform(test[['founded_year']]).ravel()

def make_X(df, fy):
    return np.hstack([
        df[STRUCTURED].values.astype('float32'),
        fy.reshape(-1, 1).astype('float32')
    ])

X_train = make_X(train, fy_train)
X_val   = make_X(val,   fy_val)
X_test  = make_X(test,  fy_test)

# Encodage des labels
le = LabelEncoder()
le.fit(train['sector'])
y_train = le.transform(train['sector'])
y_val   = le.transform(val['sector'])
y_test  = le.transform(test['sector'])
n_classes = len(le.classes_)

# Poids de classes pour compenser le déséquilibre
cw = compute_class_weight('balanced', classes=np.arange(n_classes), y=y_train)
class_weight_dict = dict(enumerate(cw))

baseline = np.bincount(y_train).max() / len(y_train)
print(f'Dimensions : {X_train.shape}')
print(f'Baseline majoritaire : {baseline:.4f}  (prédire toujours \'Other\')')

## 4. Architecture MLP

```
Input (20 features)
    → Dense(256, relu) → BatchNorm → Dropout(0.4)
    → Dense(128, relu) → BatchNorm → Dropout(0.3)
    → Dense(64,  relu) →             Dropout(0.2)
    → Dense(20, softmax)
```

| Composant | Rôle |
|-----------|------|
| BatchNormalization | Stabilise l'entraînement, accélère la convergence |
| Dropout | Régularisation — réduit le surapprentissage |
| class_weight | Compense le déséquilibre fort de la classe 'Other' |
| Softmax | Sortie probabiliste multi-classe |

In [ ]:
def build_mlp(input_dim, n_classes):
    inp = Input(shape=(input_dim,), name='structured_input')
    x   = layers.Dense(256, activation='relu')(inp)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.4)(x)
    x   = layers.Dense(128, activation='relu')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(64, activation='relu')(x)
    x   = layers.Dropout(0.2)(x)
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)
    return Model(inputs=inp, outputs=out)

model = build_mlp(X_train.shape[1], n_classes)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Afficher uniquement les paramètres
total     = model.count_params()
trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
non_train = total - trainable
print(f'Total params: {total:,} ({total*4/1024:.2f} KB)')
print(f'Trainable params: {trainable:,} ({trainable*4/1024:.2f} KB)')
print(f'Non-trainable params: {non_train:,} ({non_train*4/1024:.2f} KB)')

## 5. Entraînement

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=0
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        verbose=0
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=256,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=0   # silencieux — les résultats sont affichés ci-dessous
)

best_epoch = int(np.argmax(history.history['val_accuracy'])) + 1
best_val   = max(history.history['val_accuracy'])
n_epochs   = len(history.history['accuracy'])

print(f'Epoch {best_epoch}/{n_epochs} — val_accuracy: {best_val:.4f} (meilleur)')
print(f'Epoch {n_epochs}/{n_epochs} — early stopping (patience=5)')
print(f'\nEpochs entraînés : {n_epochs}')
print(f'Meilleur val_accuracy : {best_val:.4f}  (epoch {best_epoch})')

## 6. Courbes d'apprentissage

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(history.history['accuracy'],     label='Train', color='steelblue', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Val',   color='coral',     linewidth=2)
ax1.axvline(best_epoch - 1, color='green', linestyle='--', alpha=0.6, label=f'Best epoch ({best_epoch})')
ax1.set_title('Accuracy par epoch', fontsize=12)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend()

ax2.plot(history.history['loss'],     label='Train', color='steelblue', linewidth=2)
ax2.plot(history.history['val_loss'], label='Val',   color='coral',     linewidth=2)
ax2.set_title('Loss par epoch', fontsize=12)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Cross-Entropy Loss')
ax2.legend()

plt.suptitle('MLP — Courbes d\'entraînement', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 7. Évaluation sur le test set

In [ ]:
_, val_acc  = model.evaluate(X_val,  y_val,  verbose=0)
_, test_acc = model.evaluate(X_test, y_test, verbose=0)
preds = np.argmax(model.predict(X_test, verbose=0), axis=1)

print(f'Baseline majoritaire : {baseline:.4f}')
print(f'Accuracy validation  : {val_acc:.4f}')
print(f'Accuracy test        : {test_acc:.4f}')
print()
print(classification_report(y_test, preds, target_names=le.classes_))

In [ ]:
# F1 par classe
report  = classification_report(y_test, preds, target_names=le.classes_, output_dict=True)
classes = [k for k in report if k not in ('accuracy','macro avg','weighted avg')]
f1s     = [report[c]['f1-score'] for c in classes]

colors = ['#e74c3c' if f < 0.10 else '#f39c12' if f < 0.20 else '#2ecc71' for f in f1s]

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(classes, f1s, color=colors, edgecolor='black')
ax.set_title('F1-score par secteur — MLP (features structurées)', fontsize=12)
ax.set_ylabel('F1-score')
ax.set_ylim(0, 0.4)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 8. Comparaison ML vs DL

In [ ]:
results = [
    ('Baseline majoritaire', 'Baseline', '—',                      baseline),
    ('Naive Bayes (ML)',     'ML',       'Nom (char n-grams)',      0.5009),
    ('MLP (DL)',             'DL',       'Features structurées',    test_acc),
]
df_res = pd.DataFrame(results, columns=['Modèle','Type','Input','Accuracy test'])
print(df_res.to_string(index=False))

In [ ]:
labels = ['Baseline\n(majoritaire)', 'Naive Bayes\n(ML, nom)', 'MLP\n(DL, structuré)']
accs   = [baseline, 0.5009, test_acc]
colors = ['#95a5a6', '#3498db', '#e67e22']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, accs, color=colors, edgecolor='black', width=0.45)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.01,
            f'{acc:.4f}', ha='center', fontsize=12, fontweight='bold')
ax.set_ylim(0, 0.65)
ax.set_ylabel('Accuracy (test set)', fontsize=11)
ax.set_title('ML vs Deep Learning — Classification de secteur', fontsize=13)
plt.tight_layout()
plt.show()

print('Observation :')
print(f'  NB (ML)  = {0.5009:.4f}  →  5× mieux que le hasard aléatoire (5%)')
print(f'  MLP (DL) = {test_acc:.4f}  →  features structurées faiblement discriminantes pour le secteur')
print()
print('Conclusion :')
print('  Le nom de la startup contient un signal morphologique fort ("bio", "edu", "game").')
print('  Le financement et le pays seuls ne permettent pas d\'identifier le secteur.')

## 9. Sauvegarde du modèle

In [ ]:
model.save('mlp_sector_model.keras')
joblib.dump(le,     'mlp_label_encoder.pkl')
joblib.dump(scaler, 'mlp_scaler.pkl')

print('Modèle sauvegardé  → mlp_sector_model.keras')
print('LabelEncoder       → mlp_label_encoder.pkl')
print('Scaler             → mlp_scaler.pkl')